In [ ]:
#Imports
import textwrap
from pathlib import Path
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

This codeblock takes the top_words_per_period.csv file created in tk_unsupervised_learning.ipnyb and finds the jaccard similarity for each issue between the top 20 words of one week to the next.

In [ ]:
df = pd.read_csv("../DataFiles/top_words_per_period.csv")
jaccard_scores = []
periods = df['Period'].unique()
issues = df['Issue'].unique()
for period in range(len(periods)):
    #If this is not the last period then continue
    if period < len(periods) - 1:
        period_scores = [f"{period} to {period + 1}"]
        for issue in issues:
            #For each period and issue get the top 20 words and convert to a set
            current_words = df[(df['Period'] == period) & (df['Issue'] == issue)]['Words']
            current_words_set = {word.strip() for word in current_words.iloc[0].split(',')}
            next_words  = df[(df['Period'] == period + 1) & (df['Issue'] == issue)]['Words']
            next_words_set = {word.strip() for word in next_words.iloc[0].split(',')}

            intersection = len(current_words_set.intersection(next_words_set))
            union = len(current_words_set.union(next_words_set))
            period_scores.append(intersection / union)
        jaccard_scores.append(period_scores)
jaccard_df = pd.DataFrame(jaccard_scores,columns=['Period Range','Immigration Jaccard Score',
                                                  'Foreign Policy Jaccard Score',
                                                  'Economy Jaccard Score',
                                                  'Democracy Jaccard Score',
                                                  'Abortion Jaccard Score'])
jaccard_df.to_csv("../DataFiles/jaccard_scores.csv")

This codeblock builds the events timeline chart using a preconstructed events list that
contains notable events from May 2024 - Nov 2024.

In [ ]:
df = pd.read_csv('../DataFiles/events_list.csv')

df['Date'] = pd.to_datetime(df['Date'])

fig, ax = plt.subplots(figsize=(35, 10), constrained_layout=True)
ax.set_facecolor('lightgrey')
ax.set_ylim(-2, 2)

# Create baseline timeline axis
ax.axhline(0, color="black", lw=2)
ax.plot(df['Date'], [0] * len(df['Date']), "o", color="royalblue", markersize=10, zorder=3)

# Alternating heights to prevent overlap
levels = np.tile([0.8,-0.8,0.2,-0.2,1.2,-1.3], int(np.ceil(len(df['Date'])/3)))[:len(df['Date'])]

for i, (date, event, level) in enumerate(zip(df['Date'], df['Event'], levels)):
    # Format the date to display inside the text block
    date_str = date.strftime("%b %d, %Y")

    # Wrap the long description text at 35 characters
    wrapped_event = "\n".join(textwrap.wrap(event, width=35))

    # Combine date and text (Date is bolded via matplotlib's math text styling)
    #Create newline variable so it can be interpreted by the raw string
    nl = "\n"
    full_text = rf"$\mathbf{{{date_str}}}${nl}{wrapped_event}"

    # Draw vertical stem line from the timeline dot to the text box
    ax.vlines(date, 0, level, color="gray", linestyle="--", linewidth=1)

    # Position text box above or below the line based on level sign
    va = 'bottom' if level > 0 else 'top'

    ax.text(
        date, level, full_text,
        ha='center', va=va,
        bbox={'boxstyle':"round,pad=0.5", 'fc':"whitesmoke", 'ec':"royalblue", 'lw':1.5},
        fontsize=9.5,
        color = 'black'
    )

# 4. Clean up axes and format timelines
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)

# Hide y-axis and unnecessary borders
ax.get_yaxis().set_visible(False)
for spine in ["left", "right", "top"]:
    ax.spines[spine].set_visible(False)

plt.savefig("timeline.png")
plt.show()
